In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.metrics import sparse_categorical_accuracy

In [4]:
df=pd.read_csv(r"c:\Users\nabee\Downloads\Spam_SMS.csv")
df

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5569,spam,This is the 2nd time we have tried 2 contact u...
5570,ham,Will ü b going to esplanade fr home?
5571,ham,"Pity, * was in mood for that. So...any other s..."
5572,ham,The guy did some bitching but I acted like i'd...


In [5]:
df.head()

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.tail()

,Class,Message
5569,spam,This is the 2nd time we have tried 2 contact u...
5570,ham,Will ü b going to esplanade fr home?
5571,ham,"Pity, * was in mood for that. So...any other s..."
5572,ham,The guy did some bitching but I acted like i'd...
5573,ham,Rofl. Its true to its name


In [7]:
df.columns

Index(['Class', 'Message'], dtype='object')

In [8]:
df.shape

(5574, 2)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Class    5574 non-null   object
 1   Message  5574 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [10]:
df.isnull().sum()

Class      0
Message    0
dtype: int64

In [11]:
x=df['Message'].astype(str)
y=df['Class']

In [12]:
y=y.map({'ham' :0,'spam': 1})

In [13]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [14]:
tokenizer=Tokenizer(num_words=5000)
tokenizer.fit_on_texts(x_train)

x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)

In [15]:
# padding

x_train_pad= pad_sequences(x_train_seq,maxlen=100)
x_test_pad=pad_sequences(x_test_seq,maxlen=100)

In [16]:
model=Sequential([
    Embedding(input_dim=5000,output_dim=32),
    SimpleRNN(32),
    Dense(2, activation='softmax')
])

In [17]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [18]:
early_stop=EarlyStopping(monitor='val_loss',mode='min',patience=3,restore_best_weights=True,verbose=1)

In [19]:
checkpoint=ModelCheckpoint('rnn_model.keras',
                           monitor='val_accuracy',
                           save_best_only=True,
                           verbose=1)

In [20]:
rnn=model.fit(x_train_pad,y_train,epochs=10,batch_size=32,validation_data=(x_test_pad,y_test),
          callbacks=[early_stop,checkpoint])

Epoch 1/10
139/140 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9249 - loss: 0.2461
Epoch 1: val_accuracy improved from None to 0.96771, saving model to rnn_model.keras

Epoch 1: finished saving model to rnn_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.9251 - loss: 0.2456 - val_accuracy: 0.9677 - val_loss: 0.1012
Epoch 2/10
138/140 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9853 - loss: 0.0522
Epoch 2: val_accuracy improved from 0.96771 to 0.98386, saving model to rnn_model.keras

Epoch 2: finished saving model to rnn_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9854 - loss: 0.0518 - val_accuracy: 0.9839 - val_loss: 0.0564
Epoch 3/10
139/140 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9955 - loss: 0.0205
Epoch 3: val_accuracy improved from 0.98386 to 0.99013, saving model to rnn_model.keras

Epoch 3: finished saving model to rnn_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9953 - loss: 0.0217 - val_accurac

In [21]:
print(rnn.history.keys())

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])


In [22]:
loss,accuracy =model.evaluate(x_test_pad,y_test)
print("Test Accuracy:",accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9901 - loss: 0.0387
Test Accuracy: 0.9901345372200012
